# AP 155 Lab Exercise
---
## Exercise 5: Matrices

_Instructions_: 
- **Report figure:** No report figure. Points in the report figure will be allocated to the report discussion 
- **Report discussion:**
  - What were the basis linear equations used to set up the matrix in each question of Problem 2?
  - Once you set up the matrices, what function did you use to solve the matrix equation (Could have different answers)? Check the docstring/documentation/other references and summarize the mechanism behind the function 
- **Code**: Complete the code and ensure it is functional, error-free, readable, and efficient (where needed). Include concise Markdown documentation highlighting the critical steps of your algorithm for each problem.

### Student Information
- _Full Name (Last Name, First Name)_: Ricerra, Myke Lawrence
- _Student No._: 2024-07203
- _Section_: THU-TX-1

### Grading Information (c/o Lab Instructor)
- [Rubrics description link](https://drive.google.com/file/d/1BMSlPot2Mc7XLu0eo4S8I8gLBsIadbCL/view?usp=sharing) (Note: percentages may still be tweaked)

| Criteria | Score | Subtotal |
| --- | --- | --- |
| Report figure | XX | 20 |
| Report discussion | XX | 20|
| Code readability | XX | 20 |
| Code efficiency | XX | 20 | 
| Code appropriateness  | XX | 20 | 
| **TOTAL** | XXX | 100 |

_Date and Time Scored (MM/DD/YYYY HH:MM AM/PM):_:_

---
## Section 1: Report

<!-- ### Report figure

<img src="figures/cat.jpg" alt="Alt text" width=45%> <img src="figures/cat.jpg" alt="Alt text" width=45%>


EDIT ME. See instruction above for what to include. Insert caption here, a short description what the figure illustrates. The syntax in markdown for putting a figure is `![Description](local_file_name.png)` or `<img src="/path/to_image.jpg" alt="Alt text" width=45%>`. Make sure the figure has complete elements.
-->

### Report discussion
To solve static equilibrium problems involving a mass hanging from two ropes, linear equations are formulated along the horizontal ($x$) and vertical ($y$) axes. Balancing horizontal forces yields $T_1 \cos\alpha - T_2 \cos\beta = 0$, while balancing vertical forces gives $T_1 \sin\alpha + T_2 \sin\beta = mg$. These equations form a linear system of the form $A \mathbf{x} = \mathbf{b}$, where the coefficient matrix ($A$) contains trigonometric angle functions and is multiplied by the tension vector ($\mathbf{x}$) to equal the external force vector ($\mathbf{b}$).

To determine the unknown tensions, np.linalg.solve() is fundamentally utilized given the coefficient matrix and the external force vector. In the code, this function is placed in a separate Python file, which is then imported into the notebook. Other methods can be used, such as calculating the matrix inverse (np.linalg.inv) and multiplying it on the left of the external force vector. However, this method (explicitly computing a matrix inverse followed by matrix multiplication) is computationally inefficient and causes numerical instability. Using np.linalg.solve() is much more practical in this case, as it optimizes computational efficiency and accurately evaluates force distributions in the given two-rope structural problem.

---
## Section 2: Code

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

### Problem 1: (Circuit Analysis)


Consider a long chain of resistors wired up like this:

<img src="figures/reschain.png" alt="Alt text" width=95%>

All the resistors have the same resistance~$R$.  The power rail at the top is at voltage~$V_+=5$V.  The problem is to find the voltages $V_1...V_N$ at the internal points in the circuit.

1. **Using Ohm's law and the Kirchhoff current law**, which says that the total net current flow out of (or into) any junction in a circuit must be zero, show that the voltages $V_1\ldots V_N$ satisfy the equations

$$3V_1 - V_2 - V_3 = V_+ $$
$$-V_1 + 4V_2 - V_3 - V_4 = V_+ $$
      
$$-V_{i-2} - V_{i-1} + 4V_i - V_{i+1} - V_{i+2} = 0 $$

$$-V_{N-3} - V_{N-2} + 4V_{N-1} - V_N = V_- = 0 $$
$$-V_{N-2} - V_{N-1} + 3V_N = V_- = 0 $$

Note: The assignment of $V_-$ is to make the equations symmetric

2. Express these equations in vector form~$A\vec{v} = \vec{w}$ and find the values of the matrix~$A$ and the vector~$\vec{w}$.

3. Write a program to solve for the values of the~$V_i$ when there are $N=6$ internal junctions with unknown voltages.  (Hint: All the values of $V_i$ should lie between zero and $5$V.  If they don't, something is wrong.)

4. Now repeat your calculation for the case where there are $N=10\,000$ internal junctions.  This part is not possible using standard tools like the *solve* function.  You need to make use of the fact that the matrix~$\vec{A}$ is banded.  

For N=6, we have 6 equations.
$$3V_1-V_2-V_3=V_+$$
$$-V_1+4V_2-V_3-V_4=V_+$$
$$-V_{i-2}-V_{i-1}+4V_{i}-V_{i+1}-V_{i+2}=0\quad\text{for i = 3, 4}$$
$$-V_3-V_4+4V_5-V_6=V_-=0$$
$$-V_4-V_5+3V_6=V_-=0$$
This correspond to
$$\begin{bmatrix}
3&-1&-1&0&0&0\\
-1&4&-1&-1&0&0\\
-1&-1&4&-1&-1&0\\
0&-1&-1&4&-1&-1\\
0&0&-1&-1&4&-1\\
0&0&0&-1&-1&3
\end{bmatrix}\begin{bmatrix}
V_1\\
V_2\\
V_3\\
V_4\\
V_5\\
V_6
\end{bmatrix}=\begin{bmatrix}
V_+\\
V_+\\
0\\
0\\
V_-\\
V_-
\end{bmatrix}=\begin{bmatrix}
V_+\\
V_+\\
0\\
0\\
0\\
0
\end{bmatrix}$$

In [2]:
def cacoef(N):
    diag = 4*np.ones(N)
    diag[0] = 3
    diag[N-1] = 3
    int_mat = np.diag(diag)

    ones = -1*np.ones(N)

    ones_up = np.copy(ones)
    ones_upp = np.copy(ones)
    ones_up[0] = 0
    ones_upp[0] = ones_upp[1] = 0
    up = np.roll(np.diag(ones_up),-1,0) + np.roll(np.diag(ones_upp),-2,0)

    ones_down = np.copy(ones)
    ones_downn = np.copy(ones)
    ones_down[N-1] = 0
    ones_downn[N-1] = ones_downn[N-2] = 0
    down = np.roll(np.diag(ones_down),1,0) + np.roll(np.diag(ones_downn),2,0)
    
    return int_mat+up+down

In [3]:
vPlus = 5

def casolve(N):
    w = 0*np.ones(N)
    w[0] = w[1] = vPlus
    return np.linalg.solve(cacoef(N),w)

In [4]:
print(casolve(6))

[3.7254902  3.43137255 2.74509804 2.25490196 1.56862745 1.2745098 ]


In [5]:
print(casolve(10000))

[4.99888228e+00 4.99861842e+00 4.99802841e+00 ... 1.97158611e-03
 1.38158071e-03 1.11772227e-03]


#### Problem 1 code summary

I first created a function that returns the coefficient matrix. By exploiting the pattern in the system of equations, I generalized the process to quickly construct coefficient matrices for systems with a larger number of nodes. From this matrix, the voltage at each node can be determined.

---

### Problem 2 (Statics)


Solve the following matrix problems. Set-up the linear equations that describe the problem then solve them 

Part 1: Suppose you have a hanging mass M supported by two ropes (angled by $\alpha$ and $\beta$ with respect to the ceiling). Make a function that calculates the tension on each of the ropes. Put your function in a python file and call it here in the notebook. Check that the answer by computer matches your own computations.

For $x$-axis:
$$T_1\cos\alpha=T_2\cos\beta\implies T_1\cos\alpha-T_2\cos\beta=0$$
For $y$-axis:
$$T_1\sin\alpha+T_2\sin\beta=mg$$
$$\begin{bmatrix}
\cos\alpha&-\cos\beta\\
\sin\alpha&\sin\beta
\end{bmatrix}\begin{bmatrix}
T_1 \\
T_2
\end{bmatrix}=\begin{bmatrix}
0\\
mg
\end{bmatrix}$$

In [6]:
import tension2ropes as tn

In [19]:
a = np.pi/4 #angle
b = np.pi/3 #angle
m = 1 #kg
g = 9.81 #m/s^s

extForces = np.array([0, m*g])
tensionCoef = tn.coef(a,b)
tension = tn.solve(tensionCoef, extForces)

print(tension) # in N

[5.07802966 7.18141842]


Part 2: 
Masses $m_1, m_2, m_3$, and $m_4$ lie on a $2.00$ [m] beam of negligible mass. They are located $0.20, 0.70, 1.10$, and $1.40$ [m] away from the left end of the beam. Determine the masses $m_1, m_2, m_3$, and $m_4$, given the following constraints: 

- The total mass is $8.00$ [kg]
- If a pivot is placed halfway, the beam will balance if a $1.05$ [kg] mass is placed on the right-end of the beam
- If a pivot is placed $1.20$ [m] away from the right end, then the beam would balance if a $550$ [g] mass is placed on the left end of the beam.
- If the system is split midway, the total mass on the left is $1.00$ [kg] heavier than the total mass on the right

#### Problem 2 code summary

The equations along the x- and y-axes are expressed in matrix form. With the coefficients of tension and other constants (non-tension forces), the tension in each rope can now be determined. I also created a Python file containing essential functions, which are imported and called to solve this problem.

---

## Section 3 (Notes)

### Some codes for solving matrix problems

* solving matrix equations ($\bf{A}\vec{x} = \vec{b}$) -> `np.linalg.solve`
* LU-decomposition ($\bf{A} = \bf{L}\bf{U}$) -> `scipy.linalg.lu`
* matrix inversion ($\bf{A}^{-1}$) -> `np.linalg.inv`
* QR-decomposition ($\bf{A} = \bf{Q}\bf{R}$) -> `scipy.linalg.qr`
* eigenvalue problem ($\bf{A}\vec{x} = \lambda \vec{x}$) -> `np.linalg.eig`

Solving matrix problems by themselves isn't difficult. There is an abundance of linear algebra libraries that are ready to use: from the battle-tested ones (the classic [BLAS](https://www.netlib.org/blas/) and [LAPACK](https://www.netlib.org/lapack/)), HPC tools for large problems ([ScaLAPACK](https://netlib.org/scalapack/)), to hardware-accelerated ones often with GPUs ([cuBLAS](https://docs.nvidia.com/cuda/cublas/index.html) and [cuSolver](https://docs.nvidia.com/cuda/cusolver/index.html) for NVIDIA). 

The hardest part is actually composing the matrix. How do you transform a physical problem into a linear algebra problem?

* Circuit: https://personal.math.vt.edu/embree/cmda3606/chapter2.pdf
* Embree [main notes](https://personal.math.vt.edu/embree/cmda3606notes.pdf) + [lab manual](https://personal.math.vt.edu/embree/labman.pdf)

#### Sample Usage (Testing different codes)

In [8]:
A = np.array([[1,0,2],[-2,-1,3],[0,3,5]])
B = np.array([[-2],[1],[-3]])


In [9]:
sigma_z = [[1,0],[0,-1]]
sigma_y = [[0,-1j],[1j,0]]
sigma_x = [[0,1],[1,0]]

In [10]:
ans = np.linalg.solve(sigma_x, sigma_y)
ans = np.linalg.inv(A) @ B


In [11]:
from scipy.linalg import *
p, l, u = lu(A)
p, l, u

(array([[0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]]),
 array([[ 1.        ,  0.        ,  0.        ],
        [-0.        ,  1.        ,  0.        ],
        [-0.5       , -0.16666667,  1.        ]]),
 array([[-2.        , -1.        ,  3.        ],
        [ 0.        ,  3.        ,  5.        ],
        [ 0.        ,  0.        ,  4.33333333]]))

### Importing functions from a python file

In [12]:
import samplePackage as sP

In [13]:
testMat = sP.createZerosArray(4,5)

In [14]:
testMat[1] = [1,2,3,4,5]

In [15]:
testMat

array([[0., 0., 0., 0., 0.],
       [1., 2., 3., 4., 5.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

### np.roll

In [16]:
## Useful tips. Not required to solve the problems below!

# Suppose you have a Numpy array
x = np.zeros((4,4))
x[1,2] = 5
for row in x:
    print(row)

#rint(x[0,1:3])


# I can move all values to the left/right using roll
x_roll = np.roll(x, 1) # change 1 to -1, what happens? 
#x_roll2 = np.roll(x, -1)
print(x_roll)

x_roll2 = np.roll(x, 2)
print(x_roll2)

[0. 0. 0. 0.]
[0. 0. 5. 0.]
[0. 0. 0. 0.]
[0. 0. 0. 0.]
[[0. 0. 0. 0.]
 [0. 0. 0. 5.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [5. 0. 0. 0.]
 [0. 0. 0. 0.]]
